<a href="https://colab.research.google.com/github/ChaMooKwan/SunMoon-Univ.-Machin-Learning-Project/blob/main/%EA%B8%B0%EA%B3%84%ED%95%99%EC%8A%B5%ED%94%84%EB%A1%9C%EC%A0%9D%ED%8A%B8_ko_BERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import numpy as np
import pandas as pd

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

dataset = pd.read_csv('Training_비율통합.csv')

In [ ]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 78121 entries, 0 to 78120
Data columns (total 23 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Aspect             78121 non-null  object
 1   SentimentText      78121 non-null  object
 2   SentimentPolarity  78121 non-null  int64 
 3   가격                 78121 non-null  int64 
 4   기능                 78121 non-null  int64 
 5   내구성                78121 non-null  int64 
 6   디자인                78121 non-null  int64 
 7   무게                 78121 non-null  int64 
 8   배터리                78121 non-null  int64 
 9   사이즈                78121 non-null  int64 
 10  색상                 78121 non-null  int64 
 11  소비전력               78121 non-null  int64 
 12  소음                 78121 non-null  int64 
 13  소재                 78121 non-null  int64 
 14  시간/속도              78121 non-null  int64 
 15  용량                 78121 non-null  int64 
 16  음량/음질              78121 non-null  int64

In [ ]:
#측면에 대한 정답 레이블 생성
dataset['AspectConfidence'] = 1

In [ ]:
# transformers 설치 여부 확인
!pip list | grep transformer

sentence-transformers                    5.4.1
transformers                             5.0.0


In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer

In [ ]:
#측면 및 긍/부정 분류 모델 생성
class ABSAModel(nn.Module):
    def __init__(self, model_name="skt/kobert-base-v1"):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)

        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(0.1)

        # aspect 존재 여부
        self.aspect_classifier = nn.Linear(hidden_size, 2)

        # sentiment 분류
        self.sentiment_classifier = nn.Linear(hidden_size, 3)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls_output = outputs.last_hidden_state[:, 0]

        cls_output = self.dropout(cls_output)

        aspect_logits = self.aspect_classifier(cls_output)
        sentiment_logits = self.sentiment_classifier(cls_output)

        return aspect_logits, sentiment_logits

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    "skt/kobert-base-v1"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/535 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/244 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/371k [00:00<?, ?B/s]

In [ ]:
from torch.utils.data import Dataset

#모델 학습 전용 데이터셋
class ABSADataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        review = row["SentimentText"]
        aspect = row["Aspect"]

        encoding = self.tokenizer(
            review,
            aspect,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),

            "aspect_label": torch.tensor(row["AspectConfidence"], dtype=torch.long),

            "sentiment_label": torch.tensor(row["SentimentPolarity"], dtype=torch.long)
        }

In [ ]:
import random

#데이터셋 생성
train_data = dataset[['SentimentText', 'Aspect', 'SentimentPolarity', 'AspectConfidence']]
train_data['SentimentPolarity'] = train_data['SentimentPolarity'] + 1

#시드 고정
random.seed(42)

def create_negative_aspect_samples(data):
    negative_data = []

    # 각 aspect의 positive 개수
    aspect_counts = data['Aspect'].value_counts()

    for target_aspect, count in aspect_counts.items():

        # target_aspect가 아닌 문장들만 후보
        candidate_rows = data[
            data['Aspect'] != target_aspect
        ]

        # 중복 허용 여부
        sampled_rows = candidate_rows.sample(
            n=count,
            replace=len(candidate_rows) < count,
            random_state=42
        )

        for _, row in sampled_rows.iterrows():

            negative_data.append({
                "SentimentText": row["SentimentText"],
                "Aspect": target_aspect,
                "SentimentPolarity": -1,   # mask
                "AspectConfidence": 0
            })

    return negative_data

negative_data = create_negative_aspect_samples(train_data)

#학습 셋과 합치기
train_data = pd.concat([train_data, pd.DataFrame(negative_data)], ignore_index=True)

train_data.describe()

/tmp/ipykernel_1266/2779966107.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_data['SentimentPolarity'] = train_data['SentimentPolarity'] + 1


,SentimentPolarity,AspectConfidence
count,156242.000000,156242.000000
mean,0.262266,0.500000
std,1.392849,0.500002
min,-1.000000,0.000000
25%,-1.000000,0.000000
50%,-0.500000,0.500000
75%,2.000000,1.000000
max,2.000000,1.000000


In [ ]:
aspect_sentiment_ratio = pd.crosstab(
    train_data['Aspect'],
    train_data['SentimentPolarity'],
    normalize='index'
)

print(aspect_sentiment_ratio)

SentimentPolarity   -1         0         1         2
Aspect                                              
가격                 0.5  0.046812  0.011519  0.441670
기능                 0.5  0.069012  0.014197  0.416791
내구성                0.5  0.295473  0.022635  0.181892
디자인                0.5  0.064155  0.010204  0.425640
무게                 0.5  0.170518  0.025916  0.303566
배터리                0.5  0.237315  0.011882  0.250803
사이즈                0.5  0.104983  0.016179  0.378838
색상                 0.5  0.097705  0.010590  0.391704
소비전력               0.5  0.157895  0.000000  0.342105
소음                 0.5  0.189568  0.025180  0.285252
소재                 0.5  0.310345  0.013493  0.176162
시간/속도              0.5  0.126046  0.019825  0.354129
용량                 0.5  0.179530  0.028523  0.291946
음량/음질              0.5  0.103415  0.031100  0.365485
제조일/제조사            0.5  0.089856  0.005090  0.405054
제품구성               0.5  0.180898  0.015258  0.303844
조작성                0.5  0.141408  0.019945  0.

In [ ]:
from torch.utils.data import DataLoader

#모델 학습용 데이터셋으로 변환
torch_data_set = ABSADataset(train_data, tokenizer)

train_loader = DataLoader(
    torch_data_set,
    batch_size=32,
    shuffle=True
)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

#모델 생성
model = ABSAModel().to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-5
)

#Loss 함수 생성
criterion_aspect = nn.CrossEntropyLoss()
criterion_sentiment = nn.CrossEntropyLoss()

model.safetensors:   0%|          | 0.00/369M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
print(train_data["SentimentPolarity"].unique())
print(train_data["AspectConfidence"].unique())

[ 2  0  1 -1]
[1 0]


In [ ]:
#학습 재개 시, 모델 불러올 때 사용
checkpoint = torch.load(
    "/content/drive/MyDrive/review_classifier_model.pt",
    map_location=device
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

In [ ]:
from google.colab import drive

# 구글 드라이브 마운트
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from tqdm.auto import tqdm

epochs = 1

for epoch in range(epochs):
    model.train()

    total_loss = 0

    aspect_correct = 0
    aspect_total = 0

    sentiment_correct = 0
    sentiment_total = 0

    #학습 진행 상황
    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{epochs}"
    )

    for batch in progress_bar:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        aspect_labels = batch["aspect_label"].to(device)
        sentiment_labels = batch["sentiment_label"].to(device)

        optimizer.zero_grad()

        aspect_logits, sentiment_logits = model(
            input_ids,
            attention_mask
        )

        sentiment_loss = criterion_sentiment(
                sentiment_logits[mask],
                sentiment_labels[mask]
            )

        #측면 Loss
        aspect_loss = criterion_aspect(
            aspect_logits,
            aspect_labels
        )


        #측면이 부정인 경우는 긍/부정 Loss에서 제외
        mask = (sentiment_labels != -1)

        #측면이 긍정인 경우에만 Loss 확인
        if mask.any():

            sentiment_loss = criterion_sentiment(
                sentiment_logits[mask],
                sentiment_labels[mask]
            )

        else:
            sentiment_loss = aspect_loss * 0

        loss = aspect_loss + sentiment_loss

        #역전파
        loss.backward()

        #adam 업데이트
        optimizer.step()

        total_loss += loss.item()

        #accuracy 계산(속성어)
        aspect_preds = torch.argmax(
            aspect_logits,
            dim=1
        )

        aspect_correct += (
            aspect_preds == aspect_labels
        ).sum().item()

        aspect_total += aspect_labels.size(0)

        aspect_acc = aspect_correct / aspect_total

        #accuracy 계산(감정)
        if mask.any():

            sentiment_preds = torch.argmax(
                sentiment_logits[mask],
                dim=1
            )

            sentiment_correct += (
                sentiment_preds == sentiment_labels[mask]
            ).sum().item()

            sentiment_total += mask.sum().item()

        sentiment_acc = (
            sentiment_correct / sentiment_total
            if sentiment_total > 0 else 0
        )

        # tqdm에 현재 loss 표시
        progress_bar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "aspect": f"{aspect_loss.item():.4f}",
            "sentiment": f"{sentiment_loss.item():.4f}",
            "a_acc": f"{aspect_acc:.4f}",
            "s_acc": f"{sentiment_acc:.4f}"
        })

    avg_loss = total_loss / len(train_loader)

    print(f"\nEpoch {epoch+1} Average Loss: {avg_loss:.4f}")
    print(f"Aspect Acc : {aspect_acc:.4f}")
    print(f"Sentiment Acc : {sentiment_acc:.4f}")

    torch.save({
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "epoch": epoch,
    }, "content/drive/MyDrive/review_classifier_ko-BERT.pt")

Epoch 1/1:   0%|          | 0/4883 [00:00<?, ?it/s]

NameError: name 'mask' is not defined

In [ ]:
from sklearn.metrics import classification_report
from tqdm.auto import tqdm

model.eval()

aspects = dataset['Aspect'].unique()

evl_dataset = dataset[['SentimentText', 'Aspect', 'SentimentPolarity', 'AspectConfidence']]
evl_dataset['SentimentPolarity'] = evl_dataset['SentimentPolarity'] + 1

#Aspect를 aspects 기준으로 index 변환
evl_dataset['Aspect'] = evl_dataset['Aspect'].apply(lambda x: aspects.tolist().index(x))

review = "몇일을 파우치 없이 들고다녀 봤는데도 무겁지 않고 괜찮았습니다."

#classification 확인용
all_aspect_preds = []
all_aspect_labels = []

all_sentiment_preds = []
all_sentiment_labels = []

progress_bar = tqdm(
    evl_dataset.itertuples(),
    total=len(evl_dataset)
)

#evl_dataset으로 classification_report 계산
with torch.no_grad():
    for idx, row in enumerate(progress_bar):
        # if idx >= 10000:
        #   break;

        aspect_pred = []
        sentiment_pred = []

        #모든 속성어에 대한 조합을 생성
        texts = [row.SentimentText] * len(aspects)
        aspect_texts = list(aspects)

        #한번에 encoding 처리
        encoding = tokenizer(
            texts,
            aspect_texts,
            return_tensors="pt",
            truncation=True,
            padding=True
        )

        input_ids = encoding["input_ids"].to(device)
        attention_mask = encoding["attention_mask"].to(device)

        aspect_logits, sentiment_logits = model(
            input_ids,
            attention_mask
        )

        aspect_pred = torch.argmax(
            aspect_logits,
            dim=-1
        ).cpu().numpy()

        sentiment_pred = torch.argmax(
            sentiment_logits,
            dim=-1
        ).cpu().numpy()

        #학습, 검증 셋에 경우 무조건 하나의 속성어만 True임
        aspect_pred_idx = np.argmax(aspect_pred)
        # final_aspect_pred = aspects[aspect_pred_idx]
        # final_sentiment_pred = ["negative", "neutral", "positive"][sentiment_pred[aspect_pred_idx]]

        #속성어 index와 긍/부정 저장
        all_aspect_preds.append(aspect_pred_idx)
        all_aspect_labels.append(row.Aspect)

        all_sentiment_preds.append(sentiment_pred[aspect_pred_idx])
        all_sentiment_labels.append(row.SentimentPolarity)

print(classification_report(
    all_sentiment_labels,
    all_sentiment_preds,
    target_names=[
        "negative",
        "neutral",
        "positive"
    ]
))

print(classification_report(
    all_aspect_labels,
    all_aspect_preds,
    target_names=aspects
))

/tmp/ipykernel_1266/3221976572.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  evl_dataset['SentimentPolarity'] = evl_dataset['SentimentPolarity'] + 1
/tmp/ipykernel_1266/3221976572.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  evl_dataset['Aspect'] = evl_dataset['Aspect'].apply(lambda x: aspects.tolist().index(x))


  0%|          | 0/78121 [00:00<?, ?it/s]

              precision    recall  f1-score   support

    negative       0.21      0.36      0.27     17343
     neutral       0.03      0.48      0.05      2458
    positive       0.76      0.04      0.08     58320

    accuracy                           0.12     78121
   macro avg       0.33      0.29      0.13     78121
weighted avg       0.62      0.12      0.12     78121

              precision    recall  f1-score   support

          가격       0.14      0.65      0.23     10852
       시간/속도       0.02      0.01      0.01      2749
          품질       0.08      0.04      0.06      5382
          기능       0.12      0.00      0.01      9368
         조작성       0.08      0.02      0.04      5866
         디자인       0.05      0.00      0.01      4606
          용량       0.01      0.01      0.01       894
          화질       0.06      0.02      0.03      4345
         편의성       0.07      0.00      0.00      5881
          소재       0.00      0.00      0.00       667
          소음       0.00 

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
